In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -qU accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [3]:
!pip install datasets==2.21.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 10.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires di

In [4]:
from datasets import load_dataset

dataset_restaurant = load_dataset("jakartaresearch/semeval-absa", name='restaurant')
dataset_laptop = load_dataset("jakartaresearch/semeval-absa", name='laptop')
dataset_fabsa = load_dataset("jordiclive/FABSA")

Generating train split:   0%|          | 0/3044 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/3048 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/7930 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1057 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1587 [00:00<?, ? examples/s]

In [5]:
from tqdm.auto import tqdm
from typing import TypedDict, Generator
from datasets import Dataset, Features, Value, ClassLabel, DatasetDict


class LabeledExample(TypedDict):
    aspect: str
    labels: str
    sentence: str


def expand_dataset_semeval(dataset: Dataset) -> Generator[LabeledExample, None, None]:
    POLARITY_SET = {'positive', 'negative', 'neutral'}
    for data in tqdm(dataset):
        sentence = data['text']  # type: ignore
        terms = data['aspects']['term']  # type: ignore
        polarities = data['aspects']['polarity']  # type: ignore

        for term, polarity in zip(terms, polarities):
            if polarity not in POLARITY_SET or term == '':
                continue
            yield {
                'aspect': term,
                'labels': polarity,
                'sentence': sentence
            }


def get_dataset_semeval(dataset: DatasetDict) -> DatasetDict:
    ds_features = Features({
        'aspect': Value('string'),
        'labels': ClassLabel(names=['positive', 'negative', 'neutral']),
        'sentence': Value('string')
    })

    ds = DatasetDict(
        {
            split_name: \
            Dataset.from_generator(
                expand_dataset_semeval,
                features=ds_features,
                gen_kwargs={'dataset': dataset[split_name]}
            )
            for split_name in ['train', 'validation']
        }
    )
    return ds


def expand_dataset_fabsa(dataset: Dataset) -> Generator[LabeledExample, None, None]:
    POLARITY_SET = {'positive', 'negative', 'neutral'}
    for data in tqdm(dataset):
        sentence = data['text']  # type: ignore

        for term, polarity in data['labels']:
            if polarity not in POLARITY_SET or term == '':
                continue
            yield {
                'aspect': term.split(": ")[-1],
                'labels': polarity,
                'sentence': sentence
            }


def get_dataset_fabsa(dataset: DatasetDict) -> DatasetDict:
    ds_features = Features({
        'aspect': Value('string'),
        'labels': ClassLabel(names=['positive', 'negative', 'neutral']),
        'sentence': Value('string')
    })

    ds = DatasetDict(
        {
            split_name: \
            Dataset.from_generator(
                expand_dataset_fabsa,
                features=ds_features,
                gen_kwargs={'dataset': dataset[split_name]}
            )
            for split_name in ['train', 'validation', 'test']
        }
    )
    return ds


In [6]:
# Load and preprocess datasets into raw (non-tokenized) format
ds_restaurant_raw = get_dataset_semeval(dataset_restaurant)
ds_laptop_raw = get_dataset_semeval(dataset_laptop)
ds_fabsa_raw = get_dataset_fabsa(dataset_fabsa)

Generating train split: 0 examples [00:00, ? examples/s]

  0%|          | 0/3044 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

  0%|          | 0/800 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

  0%|          | 0/7930 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

  0%|          | 0/1057 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

  0%|          | 0/1587 [00:00<?, ?it/s]

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("Surabhii/deberta-v3-base-neutralabsa-semeval2015-fabsa-new")
model = AutoModelForSequenceClassification.from_pretrained("Surabhii/deberta-v3-base-neutralabsa-semeval2015-fabsa-new")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/538 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [9]:
from transformers import Trainer

trainer = Trainer(
    model=model
)

In [10]:
def tokenize_function(examples):
    """This tokenizer returns the format -
    [CLS] aspect tokens [SEP] sentence tokens [SEP]
    """
    return tokenizer(examples['aspect'], examples['sentence'], padding=True, truncation=True, return_token_type_ids=False, max_length=512)

In [11]:
ds_restaurant_val = ds_restaurant_raw['validation'].map(tokenize_function, batched=True)
ds_laptop_val = ds_laptop_raw['validation'].map(tokenize_function, batched=True)
ds_fabsa_test = ds_fabsa_raw['test'].map(tokenize_function, batched=True)

Map:   0%|          | 0/1120 [00:00<?, ? examples/s]

Map:   0%|          | 0/638 [00:00<?, ? examples/s]

Map:   0%|          | 0/2812 [00:00<?, ? examples/s]

In [12]:
print("Restaurant validation:", len(ds_restaurant_val))
print("Laptop validation    :", len(ds_laptop_val))
print("FABSA test           :", len(ds_fabsa_test))

Restaurant validation: 1120
Laptop validation    : 638
FABSA test           : 2812


In [13]:
import evaluate
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)  # Ensure predictions are derived correctly

    # Load the f1 metric from evaluate
    f1 = evaluate.load('f1')

    # Calculate F1 score using the 'macro' average method
    f1_result = f1.compute(predictions=predictions, references=labels, average='macro')

    # Compute the confusion matrix
    conf_matrix = confusion_matrix(labels, predictions)

    # Return a dictionary with the metric name and its value
    return {"f1_score": f1_result['f1'], "confusion_matrix": conf_matrix, 'classification_report': classification_report(labels, predictions)}

In [14]:
label_map = {
0: "positive",
1: "negative",
2: "neutral"
}

In [15]:
def print_misclassified_examples(predictions, labels, dataset):
    misclassified_count = 0
    
    # Counters
    pos_wrong = 0
    neutral_wrong = 0
    neg_wrong = 0
    
    print("\n========== MISCLASSIFIED EXAMPLES ==========\n")
    
    for i, (pred, true) in enumerate(zip(predictions, labels)):
    
        if pred != true:
            misclassified_count += 1
    
            sentence = dataset[i]['sentence']
            aspect = dataset[i]['aspect']
    
            pred_label = label_map[pred]
            true_label = label_map[true]
    
            print(f"Example #{misclassified_count}")
            print(f"Sentence       : {sentence}")
            print(f"Aspect         : {aspect}")
            print(f"True Label     : {true_label}")
            print(f"Predicted Label: {pred_label}")
            print("-" * 80)
    
            # Count category-wise errors
    
            # True Positive -> Pred Neutral/Negative
            if true == 0 and pred in [1, 2]:
                pos_wrong += 1
    
            # True Neutral -> Pred Positive/Negative
            elif true == 2 and pred in [0, 1]:
                neutral_wrong += 1
    
            # True Negative -> Pred Positive/Neutral
            elif true == 1 and pred in [0, 2]:
                neg_wrong += 1
    
    print("\n========== MISCLASSIFICATION SUMMARY ==========\n")
    
    print(f"Total Misclassified Examples: {misclassified_count}")
    
    print(f"\nTrue Positive predicted as Neutral/Negative : {pos_wrong}")
    
    print(f"True Neutral predicted as Positive/Negative : {neutral_wrong}")
    
    print(f"True Negative predicted as Positive/Neutral : {neg_wrong}")

In [16]:
model.eval()
pred = trainer.predict(ds_restaurant_val)
restaurant_metrics = compute_metrics(
    (pred.predictions, ds_restaurant_val['labels']))
print(restaurant_metrics['classification_report'])

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.92      0.96      0.94       728
           1       0.85      0.86      0.86       196
           2       0.76      0.63      0.69       196

    accuracy                           0.89      1120
   macro avg       0.84      0.82      0.83      1120
weighted avg       0.88      0.89      0.88      1120



In [17]:
print("\nConfusion Matrix:")
print(restaurant_metrics['confusion_matrix'])


Confusion Matrix:
[[701   9  18]
 [  6 169  21]
 [ 52  21 123]]


In [18]:
restaurant_predictions = np.argmax(pred.predictions, axis=1)

print_misclassified_examples(
restaurant_predictions,
ds_restaurant_val['labels'],
ds_restaurant_val
)



========== MISCLASSIFIED EXAMPLES ==========

Example #1
Sentence       : Certainly not the best sushi in New York, however, it is always fresh, and the place is very clean, sterile.
Aspect         : place
True Label     : positive
Predicted Label: negative
--------------------------------------------------------------------------------
Example #2
Sentence       : How pretentious and inappropriate for MJ Grill to claim that it provides power lunch and dinners!
Aspect         : dinners
True Label     : negative
Predicted Label: neutral
--------------------------------------------------------------------------------
Example #3
Sentence       : Entrees include classics like lasagna, fettuccine Alfredo and chicken parmigiana.
Aspect         : Entrees
True Label     : neutral
Predicted Label: positive
--------------------------------------------------------------------------------
Example #4
Sentence       : Entrees include classics like lasagna, fettuccine Alfredo and chicken parmigiana.


In [19]:
model.eval()
pred = trainer.predict(ds_laptop_val)
laptop_metrics = compute_metrics(
    (pred.predictions, ds_laptop_val['labels']))
print(laptop_metrics['classification_report'])

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.92      0.89      0.90       341
           1       0.65      0.94      0.77       128
           2       0.76      0.56      0.64       169

    accuracy                           0.81       638
   macro avg       0.77      0.79      0.77       638
weighted avg       0.82      0.81      0.81       638



In [20]:
print("\nConfusion Matrix:")
print(laptop_metrics['confusion_matrix'])


Confusion Matrix:
[[302  15  24]
 [  2 120   6]
 [ 25  50  94]]


In [21]:
laptop_predictions = np.argmax(pred.predictions, axis=1)

print_misclassified_examples(
laptop_predictions,
ds_laptop_val['labels'],
ds_laptop_val
)


========== MISCLASSIFIED EXAMPLES ==========

Example #1
Sentence       : Other than not being a fan of click pads (industry standard these days) and the lousy internal speakers, it's hard for me to find things about this notebook I don't like, especially considering the $350 price tag.
Aspect         : price tag
True Label     : positive
Predicted Label: negative
--------------------------------------------------------------------------------
Example #2
Sentence       : No installation disk (DVD) is included.
Aspect         : installation disk (DVD)
True Label     : neutral
Predicted Label: negative
--------------------------------------------------------------------------------
Example #3
Sentence       : The baterry is very longer.
Aspect         : baterry
True Label     : positive
Predicted Label: negative
--------------------------------------------------------------------------------
Example #4
Sentence       : It has so much more speed and the screen is very sharp.
Aspect      

In [22]:
model.eval()
pred = trainer.predict(ds_fabsa_test)

fabsa_metrics = compute_metrics(
    (pred.predictions, ds_fabsa_test['labels']))
print(fabsa_metrics['classification_report'])

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.96      0.98      0.97      1825
           1       0.95      0.93      0.94       896
           2       0.86      0.82      0.84        91

    accuracy                           0.96      2812
   macro avg       0.92      0.91      0.92      2812
weighted avg       0.96      0.96      0.96      2812



In [23]:
print("\nConfusion Matrix:")
print(fabsa_metrics['confusion_matrix'])


Confusion Matrix:
[[1783   36    6]
 [  61  829    6]
 [   5   11   75]]


In [24]:
fabsa_predictions = np.argmax(pred.predictions, axis=1)

print_misclassified_examples(
fabsa_predictions,
ds_fabsa_test['labels'],
ds_fabsa_test
)


========== MISCLASSIFIED EXAMPLES ==========

Example #1
Sentence       : It's an ok app.
Aspect         : App website
True Label     : positive
Predicted Label: neutral
--------------------------------------------------------------------------------
Example #2
Sentence       : In general, it’s convenient, except for the nuances with the vault and the lack of the ability to deposit money into the account through the ATM of any bank.
Aspect         : Ease of use
True Label     : positive
Predicted Label: negative
--------------------------------------------------------------------------------
Example #3
Sentence       : I really like the app and it is very useful, especially the part of the forum. I write a lot of reviews, but I don't always find the place I'm looking for! If I put the name sometimes it doesn't come out (even famous monuments) and if I don't put the address. It's a bit confusing, it should be improved to encourage more reviews
Aspect         : App website
True Label   